# 🌲 Sparse AI: Topological Needle-In-A-Haystack (NIAH) Benchmark Sweep
### Empirical Retrieval Retention Curve & Phase Transition ("Retrieval Knee") Across Context Length $N = 4096$

This notebook runs the swept benchmark to close the loop on selective KV routing in **Sparse AI** on the primary paper model (**Meta-Llama-3.1-8B-Instruct**) or **TinyLlama-1.1B**:
1. **Control Variable**: Sweeps required matching depth $r \in [0, 1, 2, 3, 4, 5]$ across the Bruhat-Tits tree, corresponding to theoretical activation sparsities $0\%, 50\%, 75\%, 87.5\%, 93.8\%, 96.9\%$ (formally verified in `PrefixSparsity.lean`).
2. **Effective Cache Capacities**: Evaluates equivalent memory footprints $K \in [4096, 2048, 1024, 512, 256, 128]$.
3. **Metric**: Needle retrieval recall rate across **50 random positions** (early, middle, late context).
4. **Expected Curve**: Replaces isolated extreme compression failures with the standard selective KV phase transition: **>95% recall** when active budget > 20%, a sharp **retrieval knee** at ~12–15%, and **information starvation** below 10%.

In [ ]:
# Step 1: Hardware Check & Hugging Face Secrets Authentication
import os
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
gpu_name = "CPU"
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Device: {gpu_name}")
    print(f"Total VRAM: {vram_gb:.2f} GB")

# Load HF_TOKEN from Colab Secrets
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ["HF_TOKEN"] = token
        print("✅ HF_TOKEN successfully authenticated from Colab Secrets.")
    else:
        print("ℹ️ HF_TOKEN not set in Colab Secrets; using public access.")
except Exception as e:
    print(f"Running in local/standard environment: {e}")

In [ ]:
# Step 2: Clone/Pull Repository & Install Dependencies
import os
from pathlib import Path

if not Path("src/llama_surgery").exists():
    if Path("sparse-ai").exists():
        %cd sparse-ai
        !git pull origin main
    else:
        !git clone https://github.com/sneed-and-feed/sparse-ai.git
        %cd sparse-ai
else:
    !git pull origin main

# Install dependencies (including bitsandbytes for 8-bit support on T4)
!pip install -q transformers accelerate bitsandbytes matplotlib seaborn tqdm
!pip install -q -e .

## Step 3: Run Benchmark Sweeps & Peer-Review Ablations
Choose an experiment below:
* **Step 3A:** Standard NIAH Benchmark (Task-warmed router, 80 steps).
* **Step 3B (Peer Ablation 1):** Untrained Frozen Random Router (`--init_mode random --skip_warmup`). Zero NIAH training.
* **Step 3C (Peer Ablation 2):** Unsupervised WikiText-2 Warmup (`--warmup_dataset wikitext`). Zero synthetic needles; pure natural text pretraining tested zero-shot on NIAH.
* **Step 3D (Peer Ablation 3):** Natural Long-Document Benchmark — **QASPER** (Real arXiv papers with natural distractors, 3k–10k tokens).

In [ ]:
# Step 3A: Standard NIAH Benchmark Sweep
!python experiments/sweep_topological_niah.py \
    --model_id "meta-llama/Meta-Llama-3.1-8B-Instruct" \
    --context_len 4096 \
    --num_positions 50 \
    --train_steps 80

In [ ]:
# Step 3B (Peer Ablation 1): Untrained Frozen Router (Zero NIAH Training)
# Evaluates completely random hyperplane projection of frozen LLaMA 3.1 embeddings
!python experiments/sweep_topological_niah.py \
    --model_id "meta-llama/Meta-Llama-3.1-8B-Instruct" \
    --context_len 4096 \
    --num_positions 50 \
    --init_mode random \
    --skip_warmup

In [ ]:
# Step 3C (Peer Ablation 2): Unsupervised WikiText-2 Warmup -> Zero-Shot NIAH
# Trains router strictly on natural language next-token prediction (ZERO synthetic needles or templates)
!python experiments/sweep_topological_niah.py \
    --model_id "meta-llama/Meta-Llama-3.1-8B-Instruct" \
    --context_len 4096 \
    --num_positions 50 \
    --warmup_dataset wikitext \
    --train_steps 80

In [ ]:
# Step 3D (Peer Ablation 3): Natural Document Benchmark — QASPER (LongBench)
# Evaluates retrieval and question answering on real arXiv research papers (3k-10k tokens)
!python experiments/pred_longbench_adelic.py \
    --dataset qasper \
    --max_samples 20 \
    --output_dir results

In [ ]:
# Step 4: Display the Empirical Retrieval Curves & Matrices
from IPython.display import Image, display
from pathlib import Path
import json

# Generate publication-grade figures if results exist
!python experiments/render_publication_figures.py

print("===================================================================")
print("FIGURE 1: RETRIEVAL RETENTION VS. ACTIVE KV BUDGET (THE KNEE)")
print("===================================================================")
if Path("figures/retrieval_knee_exact_curve.png").exists():
    display(Image("figures/retrieval_knee_exact_curve.png"))
elif Path("figures/retrieval_knee_curve.png").exists():
    display(Image("figures/retrieval_knee_curve.png"))

print("\n===================================================================")
print("FIGURE 2: PASS/FAIL & EXACT MATCH RETRIEVAL HEATMAP MATRIX")
print("===================================================================")
if Path("figures/retrieval_exact_match_heatmap.png").exists():
    display(Image("figures/retrieval_exact_match_heatmap.png"))
elif Path("figures/retrieval_heatmap_corrected.png").exists():
    display(Image("figures/retrieval_heatmap_corrected.png"))
elif Path("figures/retrieval_heatmap.png").exists():
    display(Image("figures/retrieval_heatmap.png"))

In [ ]:
# Step 5: Summary Metrics Table & Exact Match Breakdown
from pathlib import Path
import json

!python experiments/analyze_results.py